# FORMS SDK Sample Mission

This notebook imports a custom FORMS routine, adds it to the mission as an explicit sequence segment, and executes it during one ISS-like orbit.

In [ ]:
import math
import sys
from pathlib import Path

from forms import FORMS
from forms.sequence import Segment, Sequence, SequenceRunner, compile_sequence

repo_root = next(
    path
    for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "pyproject.toml").exists()
)
venv_root = (repo_root / ".venv").resolve()
python_path = Path(sys.executable).resolve()

if venv_root not in python_path.parents:
    raise RuntimeError(
        "Select the 'Python 3.12 (darknessALP)' notebook kernel."
    )

from routines import SampleIGRF13MagneticFieldRoutine

print("Python executable:", python_path)
print("Routine module:", SampleIGRF13MagneticFieldRoutine.__module__)

## Configure the SDK mission

FORMS owns the orbit state, force model, clock, and fixed-step RK4(5) propagation.

In [ ]:
EARTH_RADIUS_KM = 6378.137
ISS_ALTITUDE_KM = 408.0
ISS_INCLINATION_DEG = 51.64
TARGET_STEP_S = 10.0

forms_handle = FORMS()
forms_handle.load_koe(
    a=EARTH_RADIUS_KM + ISS_ALTITUDE_KM,
    e=0.0,
    i=ISS_INCLINATION_DEG,
    raan=0.0,
    aop=0.0,
    ta=0.0,
    timestamp="2026:06:30:00:00:00",
)

orbit_period_s = forms_handle.orbit.period
step_count = math.ceil(orbit_period_s / TARGET_STEP_S)
fixed_step_s = orbit_period_s / step_count

forms_handle.time.set_span(
    mode="simulated",
    duration=orbit_period_s,
    units="seconds",
)
forms_handle.satellite.force_model.set_gravity("point_mass")
forms_handle.satellite.propagator_config.set_type("rk45")
forms_handle.satellite.propagator_config.set_mode(
    "simulated",
    fixed_dt=fixed_step_s,
)
forms_handle.satellite.propagator_config.set_history(True)
forms_handle.satellite.propagator_config.apply()
forms_handle.record(value=fixed_step_s, unit="seconds")
forms_handle.recording = True

print(f"Orbit period: {orbit_period_s / 60.0:.3f} min")
print(f"Fixed step: {fixed_step_s:.6f} s")

## Bind the imported routine

The routine object holds the active FORMS handle, evaluates FORMS' cached IGRF-13 model at the live GCRS position and epoch, registers its vector and magnitude through `forms.types`, and marks the variables so FORMS can export them during propagation.

In [ ]:
field_samples = []
routine_instance = SampleIGRF13MagneticFieldRoutine(
    forms_handle,
    lmax=13,
    record=True,
    samples=field_samples,
)

## Add the routine and recording segments

In this FORMS release, a `routine` segment declares attached per-step work in the sequence manifest. The runner's `tick` hook performs that work during the following `propagate` segment. We also add a `stream` segment so the mission manifest explicitly declares the CSV recording cadence used by `forms.record()`.

In [ ]:
base_sequence = compile_sequence(forms_handle, name="sdk_sample_mission")
setup_segment, propagate_segment = base_sequence.segments
routine_segment = Segment(
    verb="routine",
    params={"routines": ["SampleIGRF13MagneticFieldRoutine"]},
    label="routine - FORMS IGRF-13 magnetic field",
)
recording_segment = Segment(
    verb="stream",
    params={"interval": fixed_step_s, "unit": "seconds"},
    label="stream - record marked variables every fixed step",
)
mission_sequence = Sequence(
    name=base_sequence.name,
    segments=(setup_segment, routine_segment, recording_segment, propagate_segment),
)

mission_sequence.to_manifest()

In [ ]:
runner = SequenceRunner(
    forms_handle,
    mission_sequence,
    tick=routine_instance.step,
)
run_result = runner.run()

propagator = forms_handle.satellite.propagator
propagated_steps = len(propagator.solution_t) - 1
field_values_nt = [
    sample["magnitude_t"] * 1.0e9
    for sample in field_samples
]
field_variable = forms_handle.get_variable("B_IGRF13_GCRS")
magnitude_variable = forms_handle.get_variable("B_IGRF13_MAG")
recorder_path = Path(forms_handle.recorder.file_path) if forms_handle.recorder else None

print(f"Propagated steps: {propagated_steps}")
print(f"Routine samples: {len(field_samples)}")
print(
    f"IGRF-13 field range: {min(field_values_nt):.1f} to "
    f"{max(field_values_nt):.1f} nT"
)
print("Recorded CSV:", recorder_path)

In [ ]:
segment_verbs = [
    segment["verb"]
    for segment in mission_sequence.to_manifest()["segments"]
]

assert segment_verbs == ["setup", "routine", "stream", "propagate"]
assert propagated_steps == step_count
assert field_samples
assert field_variable is forms_handle.get_variable("B_IGRF13_GCRS")
assert magnitude_variable.unit == "T"
assert field_variable.record is True
assert recorder_path is not None and recorder_path.exists()

{
    "segments": segment_verbs,
    "routine": routine_segment.params["routines"][0],
    "steps": propagated_steps,
    "samples": len(field_samples),
    "recorded_csv": str(recorder_path),
    "registered_variables": [field_variable.name, magnitude_variable.name],
}